In [1]:
import sys
import os
sys.path += [ f'{os.environ["HOME"]}/.local/lib/python{sys.version_info.major}.{sys.version_info.minor}/site-packages' ]

# Now we can safely import atlasopenmagic
import atlasopenmagic as atom

In [2]:
import uproot # for reading .root files
import time # to measure time to analyse
import math # for mathematical functions such as square root
import awkward as ak # for handling complex and nested data structures efficiently
import numpy as np # # for numerical calculations such as histogramming
import matplotlib.pyplot as plt # for plotting
from matplotlib.ticker import MaxNLocator,AutoMinorLocator # for minor ticks
from lmfit.models import PolynomialModel, GaussianModel # for the signal and background fits
import vector #to use vectors
import requests # for HTTP access
import aiohttp # HTTP client support
import pandas as pd

In [3]:
atom.set_release('2025e-13tev-beta')

Fetching metadata for release: 2025e-13tev-beta...
Fetching datasets: 100%|██████████| 374/374 [00:00<00:00, 1488.04datasets/s]
✓ Successfully cached 374 datasets.
Active release: 2025e-13tev-beta. (Datasets path: REMOTE)


In [4]:
lumi = 36

In [5]:
# list with signal DSIDs: ee, mumu
# should tautau, bb, tt be included?
dsid_signal_list = [301209] 
dsid_background_list = [700323, 700324, 700325, 700470, 700471, 700472] # remove tt, 410219]

In [6]:
def get_xsec_weight(metadata, lumi):
    return (
        lumi * 1000
        * metadata["cross_section_pb"]
        * metadata["genFiltEff"]
        * metadata["kFactor"]
        / metadata["sumOfWeights"]
    )

def get_N_inclusive(metadata, lumi):
    return (
        lumi * 1000
        * metadata["cross_section_pb"]
        * metadata["genFiltEff"]
        * metadata["kFactor"]
    )

def get_inclusive_yield(metadata, lumi):
    return (
        lumi * 1000
        * metadata["cross_section_pb"]
        * metadata["genFiltEff"]
        * metadata["kFactor"]
    )

def calc_weight(xsec_weight, weight_arr, data):
    for variable in weight_arr:
        xsec_weight = xsec_weight * data[variable]
    return xsec_weight

In [7]:
variables = ['lep_pt', 'lep_eta', 'lep_phi', 'lep_e']

In [ ]:
def create_blackbox(signal_dsid_list, background_dsid_list, signal_nevents, background_nevents):

    columns = ["lep1_pt", "lep1_eta", "lep1_phi", "lep1_e", "lep2_pt", "lep2_eta", "lep2_phi", "lep2_e", "label"]
    df = pd.DataFrame(columns=columns)


    # calculate inclusive yield for each dsid
    yield_dict = {}
    signal_yield_sum = 0
    background_yield_sum = 0

    for dsid in signal_dsid_list:
        metadata = atom.get_metadata(dsid)
        inc_yield = get_inclusive_yield(metadata, lumi)
        yield_dict[dsid] = inc_yield
        signal_yield_sum = signal_yield_sum + inc_yield

    for dsid in background_dsid_list:
        metadata = atom.get_metadata(dsid)
        inc_yield = get_inclusive_yield(metadata, lumi)
        yield_dict[dsid] = inc_yield
        background_yield_sum = background_yield_sum + inc_yield


    # calculate number of events for the blackbox for each dsid
    nevents_per_sample_dict = {}

    for dsid in signal_dsid_list:
        nevents_per_sample = signal_nevents / signal_yield_sum * yield_dict[dsid]
        nevents_per_sample_dict[dsid] = nevents_per_sample

    for dsid in background_dsid_list:
        nevents_per_sample = background_nevents / background_yield_sum * yield_dict[dsid]
        nevents_per_sample_dict[dsid] = nevents_per_sample


    # extract event properties form trees and put them in df
    for dsid in signal_dsid_list:
            
            target_events = int(nevents_per_sample_dict[dsid])
            remaining = target_events
            collected_events = 0
    
            file_list = atom.get_urls(dsid, protocol='root', cache=False)
            print("filelist: ", file_list)
    
            #for url in atom.get_urls(dsid, protocol='root', cache=True):
            #    print("url: ", url)

            for afile in file_list:
                # Print which sample is being processed
                print(f'Processing file {afile} ({file_list.index(afile)+1}/{len(file_list)})')
                #print("afile: ", f'{afile}')
                print("target events: ", target_events)
    
                # Open file
                tree = uproot.open(afile + ":analysis")
                #print("The information stored in the tree is:", tree.keys())

                for data in tree.iterate(variables, library="ak", step_size=1000):

                    #print("entered for loop for data")

                    data = data[ak.num(data.lep_pt) == 2]

                    if len(data) == 0:
                        continue

                    if len(data) > remaining:
                        data = data[:remaining]


                    new_df = pd.DataFrame({"lep1_pt": ak.to_numpy(data.lep_pt[:, 0]), "lep1_eta": ak.to_numpy(data.lep_eta[:, 0]),
                                           "lep1_phi": ak.to_numpy(data.lep_phi[:, 0]), "lep1_e": ak.to_numpy(data.lep_e[:, 0]),
                                            "lep2_pt": ak.to_numpy(data.lep_pt[:, 1]), "lep2_eta": ak.to_numpy(data.lep_eta[:, 1]),
                                            "lep2_phi": ak.to_numpy(data.lep_phi[:, 1]), "lep2_e": ak.to_numpy(data.lep_e[:, 1]),
                                            "label": 1})

                    df = pd.concat([df, new_df], ignore_index=True)

                    collected_events += len(new_df)
                    remaining = target_events - collected_events

                    print("collected events: ", collected_events)
                    if collected_events >= target_events:
                        break
                    
                if collected_events >= target_events:
                    break

    for dsid in background_dsid_list:
            
            target_events = int(nevents_per_sample_dict[dsid])
            remaining = target_events
            collected_events = 0
    
            file_list = atom.get_urls(dsid, protocol='root', cache=False)
            print("filelist: ", file_list)
    
            #for url in atom.get_urls(dsid, protocol='root', cache=True):
            #    print("url: ", url)

            for afile in file_list:
                # Print which sample is being processed
                print(f'Processing file {afile} ({file_list.index(afile)+1}/{len(file_list)})')
                #print("afile: ", f'{afile}')
                print("target events: ", target_events)
    
                # Open file
                tree = uproot.open(afile + ":analysis")
                #print("The information stored in the tree is:", tree.keys())

                for data in tree.iterate(variables, library="ak", step_size=1000):

                    #print("entered for loop for data")

                    data = data[ak.num(data.lep_pt) == 2]

                    if len(data) == 0:
                        continue

                    if len(data) > remaining:
                        data = data[:remaining]


                    new_df = pd.DataFrame({"lep1_pt": ak.to_numpy(data.lep_pt[:, 0]), "lep1_eta": ak.to_numpy(data.lep_eta[:, 0]),
                                           "lep1_phi": ak.to_numpy(data.lep_phi[:, 0]), "lep1_e": ak.to_numpy(data.lep_e[:, 0]),
                                            "lep2_pt": ak.to_numpy(data.lep_pt[:, 1]), "lep2_eta": ak.to_numpy(data.lep_eta[:, 1]),
                                            "lep2_phi": ak.to_numpy(data.lep_phi[:, 1]), "lep2_e": ak.to_numpy(data.lep_e[:, 1]),
                                            "label": 0})

                    df = pd.concat([df, new_df], ignore_index=True)

                    collected_events += len(new_df)
                    remaining = target_events - collected_events

                    print("collected events: ", collected_events)
                    if collected_events >= target_events:
                        break
                    
                if collected_events >= target_events:
                    break


    # shuffel the data frame
    df = df.sample(frac=1, random_state=25).reset_index(drop=True)

    # split df into variables and labels
    df_variables = df.drop(columns=["label"])
    df_labels = df["label"]

    return df_variables, df_labels

In [27]:
df_var, df_lab = create_blackbox(dsid_signal_list, dsid_background_list, 1000, 8000)
print("var: ", df_var)
print("lab: ", df_lab)

filelist:  ['root://eospublic.cern.ch:1094//eos/opendata/atlas/rucio/user/egramsta/mc_301209.Pythia8EvtGen_A14MSTW2008LO_Zprime_NoInt_mumu_SSM3000.noskim.root']
Processing file root://eospublic.cern.ch:1094//eos/opendata/atlas/rucio/user/egramsta/mc_301209.Pythia8EvtGen_A14MSTW2008LO_Zprime_NoInt_mumu_SSM3000.noskim.root (1/1)
target events:  1000
collected events:  720
collected events:  1000
filelist:  ['root://eospublic.cern.ch:1094//eos/opendata/atlas/rucio/user/egramsta/mc_700323.Sh_2211_Zmumu_maxHTpTV2_BFilter.noskim.root']
Processing file root://eospublic.cern.ch:1094//eos/opendata/atlas/rucio/user/egramsta/mc_700323.Sh_2211_Zmumu_maxHTpTV2_BFilter.noskim.root (1/1)
target events:  92
collected events:  92
filelist:  ['root://eospublic.cern.ch:1094//eos/opendata/atlas/rucio/user/egramsta/mc_700324.Sh_2211_Zmumu_maxHTpTV2_CFilterBVeto.noskim.root']
Processing file root://eospublic.cern.ch:1094//eos/opendata/atlas/rucio/user/egramsta/mc_700324.Sh_2211_Zmumu_maxHTpTV2_CFilterBVeto.

In [29]:
df_var.describe()

,lep1_pt,lep1_eta,lep1_phi,lep1_e,lep2_pt,lep2_eta,lep2_phi,lep2_e
count,8998.000000,8998.000000,8998.000000,8998.00000,8998.00000,8998.000000,8998.000000,8998.00000
unique,8997.000000,8995.000000,8995.000000,8997.00000,8997.00000,8997.000000,8997.000000,8998.00000
top,84.104744,1.323635,-1.395125,41.05695,7.65458,-2.394895,-2.816759,27.80229
freq,2.000000,2.000000,2.000000,2.00000,2.00000,2.000000,2.000000,1.00000


In [30]:
df_lab.describe()

count     8998
unique       2
top          0
freq      7998
Name: label, dtype: int64